In [ ]:
import pandas as pd

# 指定文件路径
file_path = r'data/raw/shareholders\HLD_Shareholders.csv'

# 使用 pandas 读取 CSV 文件
df = pd.read_csv(file_path,error_bad_lines=False)

# 显示数据的前几行，确认数据已成功读取
print(df.head())

# 首先是筛选日期

In [ ]:
# Convert the 'ReptDt' column to datetime format
df['Reptdt'] = pd.to_datetime(df['Reptdt'])

# Filter the DataFrame for the date range from '2015-12-31' to '2016-12-31'
filtered_df = df[(df['Reptdt'] >= '2021-12-31') & (df['Reptdt'] <= '2022-12-31')]

filtered_df

# 进一步筛选股票234

In [ ]:
stock_list =[34, 403, 513, 526, 568, 581, 625, 636, 661, 681, 736, 821, 858, 869, 895, 
915, 963, 977, 988, 997, 999, 1206, 1210, 1212, 1219, 2009, 2025, 2030, 
2032, 2048, 2074, 2126, 2128, 2151, 2156, 2180, 2214, 2223, 2236, 2250, 
2253, 2258, 2268, 2271, 2281, 2287, 2291, 2294, 2415, 2430, 2432, 2462, 
2466, 2475, 2484, 2508, 2541, 2549, 2555, 2568, 2585, 2594, 2595, 2597, 
2598, 2601, 2648, 2653, 2657, 2690, 2705, 2733, 2737, 2867, 2868, 2873, 
2878, 2884, 2886, 2895, 2906, 2907, 2912, 2915, 2923, 2932, 2935, 2941, 
2949, 2968, 2971, 2972, 2975, 2977, 2978, 2987, 2988, 2990, 3008, 3009, 
3020, 3021, 3022, 3023, 3029, 3030, 3033, 3038, 600031, 600048, 600071, 
600089, 600141, 600148, 600160, 600188, 600196, 600197, 600285, 600295, 
600323, 600329, 600346, 600369, 600388, 600389, 600418, 600419, 600426, 
600452, 600456, 600476, 600498, 600499, 600520, 600521, 600529, 600557, 
600559, 600563, 600566, 600570, 600588, 600612, 600645, 600699, 600702, 
600732, 600741, 600763, 600809, 600809, 600859, 600862, 600872, 600875, 
600893, 600933, 600988, 600990, 600993, 600999, 601012, 601021, 601100, 
601126, 601156, 601198, 601211, 601233, 601515, 601595, 601601, 601607, 
601615, 601677, 601689, 601698, 601858, 601838, 603319, 603324, 603355, 
603360, 603383, 603386, 603439, 603456, 603517, 603533, 603565, 603566, 
603587, 603611, 603613, 603658, 603693, 603712, 603721, 603739, 603756, 
603776, 603785, 603787, 603801, 603825, 603848, 603860, 603871, 603881, 
603883, 603885, 603888, 603893, 603896, 603908, 603916, 603919, 603927, 
603987, 603992, 605123, 605198, 605286, 605296, 605338, 605366]

# 筛选出Stkcd在stock_list中的行
filtered_stkcd_df = filtered_df[filtered_df['Stkcd'].isin(stock_list)]

In [ ]:
filtered_stkcd_df

In [ ]:
filtered_stkcd_df[filtered_stkcd_df['Stkcd']==34]

# 对于这个表格，每只股票的Reptdt均有很多个时间，每个时间都有很多个股东S0301a（可能存在重复） ，现在筛选出每只股票在所有时间都存在的股东，提取出这些行（只提取出第一次出现的那一行）

In [ ]:
df=filtered_stkcd_df

In [ ]:
import pandas as pd

# 假设df是你的数据框
# 我们首先对每只股票(Stkcd)进行分组，并对股东(S0301a)在所有日期中的出现情况进行筛选

# 对每只股票(Stkcd)进行分组，找到每只股票所有时间都有的股东，并只保留第一次出现的行
def find_common_shareholders_first_occurrence(df):
    result = []
    
    # 对每个股票进行分组
    for stkcd, group in df.groupby('Stkcd'):
        # 找到所有时间都出现的股东
        common_shareholders = group.groupby('S0301a').filter(lambda x: x['Reptdt'].nunique() == group['Reptdt'].nunique())
        
        # 只保留第一次出现的股东
        first_occurrence = common_shareholders.drop_duplicates(subset='S0301a', keep='first')
        
        result.append(first_occurrence)
    
    # 将所有股票的结果合并
    return pd.concat(result)

# 使用函数筛选出每只股票所有时间都有的股东，并保留第一次出现的行
filtered_df = find_common_shareholders_first_occurrence(df)

# 输出结果
filtered_df

In [ ]:
import pandas as pd

# 假设filtered_df是你的原始数据框
# 查找重复股东
shareholder_group = filtered_df.groupby('S0301a')['Stkcd'].nunique()

# 过滤出出现在多个股票中的股东
duplicate_shareholders = shareholder_group[shareholder_group > 1].index

# 提取这些重复股东出现在的数据行，并配对股票序号
duplicate_df = filtered_df[filtered_df['S0301a'].isin(duplicate_shareholders)]

# 构建结果，包含股票对和股东名
result = []

for shareholder in duplicate_shareholders:
    stocks = duplicate_df[duplicate_df['S0301a'] == shareholder]['Stkcd'].unique()
    if len(stocks) > 1:
        for i in range(len(stocks)):
            for j in range(i + 1, len(stocks)):
                result.append((stocks[i], stocks[j], shareholder))

# 将结果转换为数据框
duplicate_stocks_df = pd.DataFrame(result, columns=['Stock1', 'Stock2', 'Shareholder'])

# 展示结果
duplicate_stocks_df

#  看看总股东有多少（53）

In [ ]:
len(duplicate_stocks_df['Shareholder'].unique())

In [ ]:
duplicate_stocks_df['Shareholder'].unique()

In [ ]:
duplicate_stocks_df = duplicate_stocks_df[['Stock1', 'Stock2', 'Shareholder']]

In [ ]:
duplicate_stocks_df

In [ ]:
# 使用 pd.melt() 将 Stock1 和 Stock2 合并为一列
stock_per_shareholder = pd.melt(duplicate_stocks_df, id_vars=['Shareholder'], value_vars=['Stock1', 'Stock2'], 
                                var_name='StockType', value_name='Stock')

# 去重，确保每个股东每个股票只出现一次
stock_per_shareholder = stock_per_shareholder.drop_duplicates(subset=['Shareholder', 'Stock'])

# 按股东分组，列出他们持有的股票
stocks_by_shareholder = stock_per_shareholder.groupby('Shareholder')['Stock'].apply(list)

# 展示结果
stocks_by_shareholder


In [ ]:
import pandas as pd

# 假设 stock_list 和 stocks_by_shareholder 已经定义
# stock_list 是您之前给出的股票列表
# stocks_by_shareholder 是每个股东持有的股票列表

# 提取所有股东（列名）
unique_shareholders = stocks_by_shareholder.index

# 创建一个全零矩阵，行是股票，列是股东
matrix = pd.DataFrame(0, index=stock_list, columns=unique_shareholders)

# 遍历 stocks_by_shareholder，为每个股东持有的股票在矩阵中填 1
for shareholder, stocks in stocks_by_shareholder.items():
    for stock in stocks:
        if stock in stock_list:
            matrix.at[stock, shareholder] = 1

# 返回最终构造的矩阵
matrix

In [ ]:
# # 将矩阵保存为 CSV 文件，不包括行和列标签
# output_path = r'ComNC_main/CAS2\matrix2.csv'
# matrix.to_csv(output_path, index=False, header=False)

In [ ]:
(matrix == 1).sum().sum()

# 这个矩阵用于构造股东的初始特征

In [ ]:
duplicate_stocks_df

# 现在构建第二张表，这是总的关系表

In [ ]:
# 构造矩阵的行和列，行和列均包含股票以及股东
matrix_index = list(stock_list) + list(unique_shareholders)
matrix = pd.DataFrame(0, index=matrix_index, columns=matrix_index)

# Step 1: 处理 stock_list 对应 stock_list 的关系
for _, row in duplicate_stocks_df.iterrows():
    stockA, stockB, shareholder = row['Stock1'], row['Stock2'], row['Shareholder']
    
    # 如果 stockA 和 stockB 都在 stock_list 中，构造双向关系
    if stockA in stock_list and stockB in stock_list:
        matrix.at[stockA, stockB] = 1
        matrix.at[stockB, stockA] = 1

# Step 2: 处理 stock_list 对应 unique_shareholders 的关系
for _, row in duplicate_stocks_df.iterrows():
    stockA, stockB, shareholder = row['Stock1'], row['Stock2'], row['Shareholder']
    
    # 如果 stockA 在 stock_list 且 shareholder 在 unique_shareholders 中，填充 1
    if stockA in stock_list and shareholder in unique_shareholders:
        matrix.at[stockA, shareholder] = 1
    
    # 如果 stockB 在 stock_list 且 shareholder 在 unique_shareholders 中，填充 1
    if stockB in stock_list and shareholder in unique_shareholders:
        matrix.at[stockB, shareholder] = 1

# 返回构造的矩阵
matrix

In [ ]:
(matrix == 1).sum().sum()

In [ ]:
# 创建两个列表用于存储行和列的序号
row_indices = []
column_indices = []

# 遍历矩阵，检查每个元素是否为 1，如果是 1 则记录其行和列的序号
for row_idx, (row_name, row) in enumerate(matrix.iterrows()):  # 使用 enumerate 获取行的序号
    for col_idx, (col_name, value) in enumerate(row.items()):  # 使用 enumerate 获取列的序号
        if value == 1:
            row_indices.append(row_idx )  # 行序号
            column_indices.append(col_idx )  # 列序号

# 将行序号和列序号组合成一个大列表
result = [row_indices, column_indices]

# 输出结果
result

In [ ]:
import torch

# 假设 result 是一个包含行索引和列索引的列表
# 将 result 列表转换为张量
result_tensor = torch.tensor(result, dtype=torch.int64)

# 输出转换后的张量
print(result_tensor)

In [ ]:
len(result_tensor[0])

# 注意这个3578既包含公司对公司的成对关系，也包含股东对公司的关系

In [ ]:
matrix.iloc[:173, :173]

In [ ]:
(matrix.iloc[:173, :173] == 1).sum().sum()

In [ ]:

import torch
import pandas as pd

# 假设 result 是一个包含行索引和列索引的列表
# 将 result 列表转换为张量
result_tensor = torch.tensor(result, dtype=torch.int64)

# 将张量转换为 DataFrame
df000 = pd.DataFrame(result_tensor.numpy().T, columns=['Row_Index', 'Col_Index'])

# 保存为 CSV 文件
output_path = r'ComNC_main/CAS2\firm_gudong.csv'
df000.to_csv(output_path, index=False)

print(f"保存成功: {output_path}")